# A phone camera lens with Forbes Q polynomials

A smartphone main camera is the most aspheric lens most people own: five
to seven moulded plastic elements, every surface aspheric, f/1.8 or
faster over almost 40 degrees half field, all in about 5 mm. Its last
element looks nothing like a lens from a textbook: a strongly curved
centre, a rim that bends back towards the sensor, and several inflection
points. These are the "wildly wavy" aspheres for which G. W. Forbes
introduced the Q-bfs polynomials, and this notebook designs one with them,
as a follow-on to notebook 128 (Forbes Q aspheres).

The lens: f = 4 mm, f/1.8, an image height of 3 mm (36.9° half field,
a 1/2.8" sensor with 1 µm pixels), five elements in two plastics (a
cyclo-olefin polymer of the E48R kind for the positive elements and a
fluorene polyester of the OKP kind as the flint), an IR cut filter, a total
track of at most 5.3 mm and a chief ray angle at the sensor of at most
34°.

| section | content |
|---|---|
| 1 | the specification, the plastics and the design |
| 2 | how the design was found: field and aperture as a homotopy |
| 3 | the same lens in two bases: monomials against Q-bfs |
| 4 | reading the Q coefficients: the departure spectrum |
| 5 | designing for manufacture: limits on the Q terms against image quality |
| 6 | truncation: dropping the highest terms |
| 7 | final results: Zernike wavefronts at three wavelengths, at infinity and in close focus |
| 8 | coda: Q2D freeform terms on a tilted mirror |

The surfaces are `qaspheres.QbfsSurface` (and `Q2DSurface` in section 8),
the optimizer the damped least squares of notebook 58 (lens optimization)
with the analytic Jacobians of the differential ray trace
(`ray_derivatives`); the operands for the sensor (chief ray angle,
distortion, lateral colour, thicknesses sampled across the aperture) are
in `pyoptics.phone_lens`.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from pyoptics import raytrace as rt
from pyoptics import lens_optimization as lo
from pyoptics import qaspheres as qa
from pyoptics import phone_lens as pl
from pyoptics import off_axis as oa

plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "figure.facecolor": "white",
                     "axes.facecolor": "#fcfcfb", "savefig.facecolor": "white",
                     "axes.grid": True, "grid.color": "0.88", "grid.linewidth": 0.6,
                     "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False})
BLUE, RED, GREEN, ORANGE, PURPLE, GREY = "#2a7ab9", "#d6604d", "#5aa469", "#e0a000", "#8e5ba8", "0.45"
WLS = pl.WAVELENGTHS
LENS = pl.LENS_SURFACES
NAMES = ["L1 front", "L1 rear", "L2 front", "L2 rear", "L3 front", "L3 rear",
         "L4 front", "L4 rear", "L5 front", "L5 rear"]
t_start = time.time()

## 1. The specification, the plastics and the design

Only two families of optical plastics are moulded in volume: the
cyclo-olefin polymers (COP, COC; $n_d \approx 1.53$–$1.545$, $V_d \approx 56$),
the "crown", and polycarbonate or fluorene polyesters ($n_d$ 1.58–1.67,
$V_d$ 19–30), the "flint". The index contrast is small, so the powers are
large and the aspheres do much of the work a glass lens would do with
more elements. The classical five element layout is

- **L1**, positive, COP, with the aperture stop on its front surface
  (a short lens needs the stop in front);
- **L2**, negative, the high dispersion flint: axial colour;
- **L3**, a weak corrector;
- **L4**, a positive meniscus bent towards the sensor;
- **L5**, a negative element close to the sensor that flattens the
  field and bends the chief rays back so that they reach the sensor at
  an angle its micro lenses accept.

The design below came out of the procedure of section 2. Every lens
surface is a Q-bfs surface: radius of the best fit sphere, normalization
radius $r_\mathrm{norm}$ and eight coefficients $a_0 \dots a_7$ (ten on the strongly curved L4 rear and L5, which needed a larger normalization radius late in the design).

In [ ]:
lens = pl.phone_lens()
full = lens.max_field
p = lens.paraxial()
rims = pl.clear_apertures(lens)
print("efl {:.3f} mm, f/{:.2f}, half field {:.1f}°, image height {:.2f} mm, TTL {:.3f} mm".format(
    p.efl, p.efl/(2*p.ep_radius), np.degrees(full), p.efl*np.tan(full), lo.total_length(lens)))
print()
print("surface    R_bfs    R_vertex  thickness  material        r_norm  clear ap.   a_0 ... a_7 [µm]")
for i, S in enumerate(lens.surfaces):
    mat = "COP" if S.material == pl.PLASTICS["COP"] else "OKP" if S.material == pl.PLASTICS["OKP"] \
        else "filter" if S.material == pl.IR_FILTER else "air"
    if isinstance(S, qa.QbfsSurface):
        print("{:9s} {:8.3f}  {:8.3f}   {:7.3f}    {:6s}         {:5.2f}   {:6.3f}    {}".format(
            NAMES[i], S.R, 1/S.c, S.thickness, mat, S.norm_radius, rims[i],
            " ".join("{:+6.1f}".format(a*1e3) for a in S.q)))
    else:
        print("{:9s} {:>8s}  {:>8s}   {:7.3f}    {:6s}".format(
            "filter" if i == 10 else "", "inf", "", S.thickness, mat))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
lens.draw(ax, fields=[0.0, 0.5*full, 0.85*full, full], n_rays=7,
          colors=[BLUE, GREEN, ORANGE, RED])
ax.set_aspect("equal")
ax.set_title("f = 4 mm f/1.8 phone camera lens: five plastic elements, IR filter, sensor")
fig.tight_layout()

The last element is the signature of a phone lens: its vertex radius is
far from the radius of the best fit sphere, and the coefficients of L5
are ten times those of L1. The checks a phone lens has to pass:

In [ ]:
cra = pl.chief_ray_angle((0.25, 0.5, 0.7, 0.85, 1.0))(lens)
dist = pl.distortion((0.25, 0.5, 0.7, 0.85, 1.0))(lens)
print("field                 " + "  ".join("{:6.2f}".format(f) for f in (0.25, 0.5, 0.7, 0.85, 1.0)))
print("chief ray angle [°]   " + "  ".join("{:6.1f}".format(v) for v in cra))
print("distortion [%]        " + "  ".join("{:6.2f}".format(100*v) for v in dist))
glass = pl.min_thickness([0, 2, 4, 6, 8], rims)(lens)
air = pl.min_thickness([1, 3, 5, 7], rims)(lens)
print("thinnest element (axis to rim) [mm]: " + ", ".join("L{} {:.3f}".format(k + 1, t) for k, t in enumerate(glass)))
print("thinnest air gap [mm]:               " + ", ".join("L{}-L{} {:.3f}".format(k + 1, k + 2, t) for k, t in enumerate(air)))

## 2. How the design was found

There is no patent behind this lens. The starting point was the layout
of section 1 with rough spherical radii for f ≈ 4 mm, and a direct
optimization from it fails: at 37° the edge rays miss the rear elements
altogether. A spherical L5 cannot have both the strong central power and
the flat rim that the field needs, so a phone lens cannot be designed with
spheres first and aspheres later.

The working procedure was a **homotopy in the field**:

1. Make every lens surface a Q-bfs surface with zero coefficients, with
   $r_\mathrm{norm}$ about the expected clear aperture.
2. Optimize at 40 % of the field, where every ray traces; the first three
   Q terms per surface are enough there.
3. Widen the field in steps of 5 %, starting each step from the previous
   solution, and allow more Q terms as the field grows (up to eight). A
   step that would lose rays is halved. Damped least squares only finds
   the nearest minimum, so each step must be small enough that the old
   solution still traces at the new field.
4. Two guard rails were needed. The radius of each best fit sphere is
   bounded to more than 1.1 $r_\mathrm{norm}$, or the surface stops existing
   at the rim (the first attempt broke down at 50 % field that way). And
   the chief ray angle limit is in the merit function from 70 % field on,
   **scaled with the field** ($\tan\theta_\mathrm{CRA} \propto$ field):
   imposed only at full field, it fought the form the lens had already
   found and the optimizer stalled.

The merit function: the wavefront on 4 rings at five fields and three
wavelengths (F, d, C) against the chief ray of each, the focal length,
the lateral colour and soft limits on distortion and chief ray angle as
operands; the total track and the thicknesses (sampled from the axis to
the rim of the real beam, which catches the thin zones of the wavy
elements) as constraints. Each step took about 15 minutes with the
analytic Jacobian. The weakness of the method is that the lens inherits
the form it settled into at small field; a different start or a global
search (notebook 154, global optimization and saddle points) may find a
better one.

## 3. The same lens in two bases

To compare the bases on the same problem, we start both from one lens:
the design with its Q-bfs coefficients truncated to $a_0 \dots a_4$
(the rest set to zero), so that the high orders have to be found again.
For the monomial run the same surfaces are refit as even aspheres: the
vertex curvature on a paraboloid ($K = -1$, which has a sag at every
radius) plus $A_4 r^4 \dots A_{16} r^{16}$, seven terms, the usual form of
a phone lens patent, fitted out to the clear aperture (or $r_\mathrm{norm}$,
whichever is larger). Both starts are the same lens to a fraction of a
micrometre. Both runs vary the radii, thicknesses and eight (Q)
or seven (monomial) aspheric terms per surface with the same operands,
constraints and number of iterations; the monomial variables are scaled
as the sag $A_k r_\mathrm{norm}^k$ at the rim, the best scaling for them.

In [ ]:
def truncated(system, n_keep):
    s = system.copy()
    for i in LENS:
        q = list(s.surfaces[i].q)
        s.surfaces[i].q = tuple(q[:n_keep] + [0.0]*(len(q) - n_keep))
    return s


def merit_parts(system, cra_max=34.0):
    rims = pl.clear_apertures(system)
    ops = [lo.paraxial_target("efl", pl.EFL, 0.002),
           lo.wavefront(fields=(0, .5, .7, .85, 1.0), wavelengths=WLS, rings=(0.35, 0.7, 0.9, 1.0),
                        spokes=8, tolerance=0.05),
           pl.lateral_colour(),
           lo.Constraint(pl.chief_ray_angle(), upper=cra_max, tolerance=0.5),
           lo.Constraint(pl.distortion(), lower=-0.025, upper=0.025, tolerance=2e-3)]
    cons = [lo.Constraint(lo.total_length, upper=5.3, tolerance=0.01),
            lo.Constraint(pl.min_thickness([0, 2, 4, 6, 8], rims), lower=0.25, tolerance=0.005),
            lo.Constraint(pl.min_thickness([1, 3, 5, 7], rims), lower=0.03, tolerance=0.005)]
    return ops, cons


NR = [lens.surfaces[i].norm_radius for i in LENS]
common = [lo.Thickness(i) for i in (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11)]
rims = pl.clear_apertures(lens)
fit_radii = list(np.maximum(1.02*rims[:10], NR)) + [0.0, 0.0]
start_q = truncated(lens, 5)
start_m, fit_err = pl.to_monomials(start_q, rims=fit_radii)
print("largest sag error of the monomial refits of the start: {:.3f} µm".format(fit_err.max()*1e3))
for name, s0 in (("Q-bfs start", start_q), ("monomial start", start_m)):
    print("{:15s} rms wavefront at 587 nm, fields 0/0.5/0.7/0.85/1: ".format(name)
          + " ".join("{:.3f}".format(s0.rms_opd(f*full)) for f in (0, .5, .7, .85, 1.0)) + " waves")
full_m, full_err = pl.to_monomials(lens, rims=fit_radii)
print("the final design itself as seven monomials: sag errors up to {:.1f} µm (L5), rms wavefront ".format(
    full_err.max()*1e3) + " ".join("{:.2f}".format(full_m.rms_opd(f*full)) for f in (0, .5, .7, .85, 1.0)) + " waves")
variables = {
    "Q-bfs": [lo.Curvature(i, bounds=(-1/(1.1*NR[i]), 1/(1.1*NR[i]))) for i in LENS] + common
             + [qa.QCoefficient(i, m) for i in LENS for m in range(8)],
    "monomials": [lo.Curvature(i) for i in LENS] + common
                 + [lo.Aspheric(i, k, NR[i]) for i in LENS for k in range(4, 18, 2)]}
N_IT = 10
runs = {}
for name, s0 in (("Q-bfs", start_q), ("monomials", start_m)):
    ops, cons = merit_parts(s0)
    opt = lo.DampedLeastSquares(s0, variables[name], ops, constraints=cons, jac="analytic")
    J = opt.jacobian()[0]
    t0 = time.time()
    res = opt.run(max_iterations=N_IT)
    runs[name] = dict(opt=opt, J0=J, history=[h["merit"] for h in res.history] + [opt.merit()],
                      time=time.time() - t0, message=res.message)
    print("{:9s}: merit {:.3g} -> {:.3g} in {} iterations ({:.0f} s), {}".format(
        name, runs[name]["history"][0], runs[name]["history"][-1], len(res.history),
        runs[name]["time"], res.message))

In [ ]:
def asphere_block(name):
    # columns of the aspheric variables, each scaled to unit norm
    r = runs[name]
    n0 = len(LENS) + len(common)
    J = np.nan_to_num(r["J0"][:, n0:]*r["opt"]._scale[n0:])
    return J/np.maximum(np.linalg.norm(J, axis=0), 1e-300)


fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for name, col in (("Q-bfs", GREEN), ("monomials", RED)):
    axes[0].semilogy(runs[name]["history"], "o-", color=col, lw=1.8, label=name)
    sv = np.linalg.svd(asphere_block(name), compute_uv=False)
    axes[1].semilogy(sv/sv[0], ".-", color=col, label="{}: cond {:.1e}".format(name, sv[0]/sv[-1]))
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("merit function")
axes[0].set_title("Convergence from the same start")
axes[0].legend()
axes[1].set_xlabel("index")
axes[1].set_ylabel("singular value / largest")
axes[1].set_title("Aspheric columns of the Jacobian (unit norm)")
axes[1].legend(fontsize=9)
# cosine between neighbouring orders on L5 rear
for name, col, mk in (("Q-bfs", GREEN, "s"), ("monomials", RED, "o")):
    B = asphere_block(name)
    k = 7 if name == "monomials" else 8
    blk = B[:, 9*k:10*k]
    cos = [abs(blk[:, j] @ blk[:, j + 1]) for j in range(k - 1)]
    axes[2].plot(range(1, k), cos, mk + "-", color=col, label=name)
axes[2].set_xlabel("order j (columns j-1 and j)")
axes[2].set_ylabel("|cos| between neighbouring columns")
axes[2].set_title("L5 rear: how alike neighbouring terms look to the optimizer")
axes[2].set_ylim(0, 1.02)
axes[2].legend()
fig.tight_layout()

**Coefficients.** The monomial design hides its surfaces in coefficients
of alternating sign that cancel each other; the Q coefficients are the
departures themselves:

In [ ]:
sm = runs["monomials"]["opt"].system
sq = runs["Q-bfs"]["opt"].system
for i in (8, 9):
    S = sm.surfaces[i]
    print("{} (monomials): R = {:.4f}, K = -1, ".format(NAMES[i], S.R)
          + ", ".join("A{} = {:+.4e}".format(4 + 2*k, a) for k, a in enumerate(S.aspheric)))
    r = NR[i]
    terms = [a*r**(4 + 2*k) for k, a in enumerate(S.aspheric)]
    print("    their sags at r_norm [µm]: " + ", ".join("{:+.1f}".format(t*1e3) for t in terms)
          + "   sum {:+.1f}".format(sum(terms)*1e3))
    print("{} (Q-bfs):     R_bfs = {:.4f}, a = ".format(NAMES[i], sq.surfaces[i].R)
          + ", ".join("{:+.2f}".format(a*1e3) for a in sq.surfaces[i].q) + " µm")
for name in ("Q-bfs", "monomials"):
    s = runs[name]["opt"].system
    print("{:9s}: rms wavefront at 587 nm, fields 0/0.5/0.7/0.85/1: ".format(name)
          + " ".join("{:.3f}".format(s.rms_opd(f*full)) for f in (0, .5, .7, .85, 1.0)) + " waves")

## 4. Reading the Q coefficients: the departure spectrum

Each Q-bfs coefficient is, to a good approximation, the rms slope
contribution of its term times $r_\mathrm{norm}$; with the slope
orthonormality the coefficients of one surface add in quadrature to its
rms slope, $\sqrt{\sum a_m^2}/r_\mathrm{norm}$. Plotted as a spectrum per
surface they show at once which surfaces carry the asphericity and how
fast it decays with order:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
NQ = max(len(lens.surfaces[i].q) for i in LENS)
M = np.array([list(lens.surfaces[i].q) + [0.0]*(NQ - len(lens.surfaces[i].q)) for i in LENS])*1e3
im = axes[0].imshow(np.log10(np.abs(M) + 1e-3), cmap="viridis", aspect="auto", vmin=-1, vmax=3)
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(NAMES)
axes[0].set_xlabel("order m")
axes[0].set_title("log10 |a_m| [µm]")
for (i, m), v in np.ndenumerate(M):
    axes[0].text(m, i, "{:.0f}".format(v) if abs(v) >= 1 else "{:.1f}".format(v),
                 ha="center", va="center", fontsize=7, color="w" if abs(v) < 100 else "k")
fig.colorbar(im, ax=axes[0])
for i, col in zip((0, 3, 6, 7, 8, 9), (BLUE, GREEN, ORANGE, PURPLE, RED, GREY)):
    S = lens.surfaces[i]
    r = np.linspace(0, S.norm_radius, 300)
    dep = S.departure(r)*np.sqrt(1 - S.c_bfs**2*r**2)
    axes[1].plot(r/S.norm_radius, dep*1e3, color=col, lw=1.8, label=NAMES[i])
axes[1].set_xlabel("u = r / r_norm")
axes[1].set_ylabel("normal departure from the bfs [µm]")
axes[1].set_title("Departures from the best fit spheres")
axes[1].legend(fontsize=8)
fig.tight_layout()
print("surface     max departure   rms slope   share of a_0..a_2 in the sum of squares")
for i in LENS:
    S = lens.surfaces[i]
    r = np.linspace(0, S.norm_radius, 2001)
    dep = S.departure(r)*np.sqrt(1 - S.c_bfs**2*r**2)
    q = np.array(S.q)
    print("{:9s}   {:9.1f} µm   {:6.1f} mrad   {:5.1f} %".format(
        NAMES[i], np.abs(dep).max()*1e3, S.rms_slope()*1e3, 100*np.sum(q[:3]**2)/np.sum(q**2)))

## 5. Designing for manufacture

The moulds of phone lenses are diamond turned and measured with
profilometers, not interferometers, so the limits are on slope and on
the fine structure that the tool can cut and the mould can fill rather
than on the departure from a sphere. Because the Q-bfs coefficients are
orthonormal in slope, such limits are simple functions of the
coefficients:

- the **rms slope** of a surface is $\sqrt{\sum_m a_m^2}/r_\mathrm{norm}$;
- the **high order content**, $\sqrt{\sum_{m \ge 4} a_m^2}$, measures the
  mid spatial frequency ripple, which costs cutting time, mould filling
  and scattered light.

With monomials neither is a simple function of the coefficients. Here
we cap the high order content of every surface at a value $h$ and
re-optimize, for a few values of $h$, and watch the image quality:

In [ ]:
def high_order(i, m0=4):
    def f(system):
        q = np.array(system.surfaces[i].q)
        return np.sqrt(np.sum(q[m0:]**2))
    f.__name__ = "high{}".format(i)
    return f


def rms_field(system, wl=WLS[1]):
    s = system.copy()
    s.wavelength = wl
    return np.array([s.rms_opd(f*full) for f in (0, .5, .7, .85, 1.0)])


limits = [None, 50e-3, 20e-3, 5e-3]
tradeoff = []
for h in limits:
    if h is None:
        tradeoff.append((h, lens, rms_field(lens)))
        continue
    ops, cons = merit_parts(lens)
    cons = cons + [lo.Constraint(high_order(i), upper=h, tolerance=1e-4) for i in LENS]
    V = variables["Q-bfs"]
    opt = lo.DampedLeastSquares(lens, V, ops, constraints=cons, jac="analytic")
    opt.run(max_iterations=6)
    tradeoff.append((h, opt.system, rms_field(opt.system)))
    print("h = {:5.1f} µm: rms wavefront {} waves".format(
        h*1e3, " ".join("{:.3f}".format(v) for v in tradeoff[-1][2])))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.3))
fr = np.array([0, .5, .7, .85, 1.0])
for (h, s, w), col in zip(tradeoff, (GREY, BLUE, GREEN, RED)):
    lab = "free" if h is None else "high orders ≤ {:.0f} µm".format(h*1e3)
    axes[0].plot(fr, w, "o-", color=col, lw=1.8, label=lab)
    hi = [np.sqrt(np.sum(np.array(s.surfaces[i].q[4:])**2))*1e3 for i in LENS]
    axes[1].semilogy(range(10), np.maximum(hi, 1e-3), "o-", color=col, label=lab)
axes[0].axhline(0.07, color="k", ls=":", lw=1)
axes[0].set_xlabel("field")
axes[0].set_ylabel("rms wavefront at 587 nm [waves]")
axes[0].set_title("Image quality against the high order limit")
axes[0].legend(fontsize=9)
axes[1].set_xticks(range(10))
axes[1].set_xticklabels(NAMES, rotation=45, ha="right")
axes[1].set_ylabel("sqrt(sum a_m², m ≥ 4) [µm]")
axes[1].set_title("High order content per surface")
fig.tight_layout()

## 6. Truncation

Orthogonality has a second practical consequence: dropping the highest
terms changes the surface by exactly those terms, and the lens degrades
gracefully. With monomials the terms are only meaningful together, so
dropping the highest one leaves its large partners uncompensated. We
truncate both final designs of section 3 to their first $k$ aspheric
terms (no re-optimization, but refocused) and compare:

In [ ]:
def truncate_mono(system, n_keep):
    s = system.copy()
    for i in LENS:
        A = list(s.surfaces[i].aspheric)
        s.surfaces[i].aspheric = tuple(A[:n_keep] + [0.0]*(len(A) - n_keep))
    return s


def refocused_rms(system):
    # best image distance for the field average, then the rms at 0.7 field and on axis
    best = None
    for dz in np.linspace(-0.02, 0.02, 9):
        s = system.copy()
        s.surfaces[-1].thickness += dz
        with np.errstate(all="ignore"):
            w = np.array([s.rms_opd(f*full) for f in (0, .7)])
        if np.all(np.isfinite(w)) and (best is None or w.mean() < best.mean()):
            best = w
    return np.full(2, np.nan) if best is None else best


kq = list(range(3, 9))
km = list(range(2, 8))
tq = [refocused_rms(truncated(sq, k)) for k in kq]
tm = [refocused_rms(truncate_mono(sm, k)) for k in km]
fig, ax = plt.subplots(figsize=(8, 4.3))
ax.semilogy(kq, [t[0] for t in tq], "s-", color=GREEN, lw=2, label="Q-bfs, axis")
ax.semilogy(kq, [t[1] for t in tq], "s--", color=GREEN, lw=2, label="Q-bfs, 0.7 field")
ax.semilogy(km, [t[0] for t in tm], "o-", color=RED, lw=2, label="monomials, axis")
ax.semilogy(km, [t[1] for t in tm], "o--", color=RED, lw=2, label="monomials, 0.7 field")
ax.set_xlabel("aspheric terms kept per surface")
ax.set_ylabel("rms wavefront at 587 nm [waves]")
ax.set_title("Truncating the aspheres (refocused, not re-optimized)")
ax.legend(fontsize=9)
fig.tight_layout()
for k, t in zip(kq, tq):
    print("Q-bfs, {} terms: {:.3f} / {:.3f} waves".format(k, *t))
for k, t in zip(km, tm):
    print("monomials, {} terms: {:.3f} / {:.3f} waves".format(k, *t))

## 7. Final results: Zernike wavefronts

The wavefront of the final design as Fringe Zernike coefficients
(`System.zernike()`, in waves of each wavelength, over the entrance pupil;
$Z_1$ piston, $Z_2, Z_3$ tilt, $Z_4$ defocus, $Z_5, Z_6$ astigmatism,
$Z_7, Z_8$ coma, $Z_9$ spherical aberration, then the higher orders), at
the three design wavelengths and five fields, with the image plane fixed
where the optimization put it.

A phone camera lens has one focal length, so there are no zoom
positions; it focuses by moving the whole lens with a voice coil motor.
Close focus is the second configuration: the object at 100 mm, the lens
moved away from the sensor so that the field is in focus again (the
lens itself unchanged).

In [ ]:
ZN = ["Z{}".format(j) for j in range(1, 17)]
FIELDS = (0.0, 0.5, 0.7, 0.85, 1.0)


def zernike_table(system, title):
    print(title)
    print("  λ [nm] field  rms  " + " ".join("{:>6s}".format(z) for z in ZN[3:16]))
    out = {}
    for wl in WLS:
        s = system.copy()
        s.wavelength = wl
        for f in FIELDS:
            z = s.zernike(f*s.max_field, n_terms=16, N=48)
            rms = s.rms_opd(f*s.max_field, N=48)
            out[wl, f] = (z, rms)
            print("  {:5.0f}  {:5.2f}  {:5.3f} ".format(wl*1e6, f, rms)
                  + " ".join("{:+6.3f}".format(v) for v in z[3:16]))
    return out


Z_inf = zernike_table(lens, "Object at infinity (Fringe Zernikes Z4...Z16 [waves], rms without tilt)")

In [ ]:
def close_focus(system, distance=100.0):
    # finite object at `distance` from the first vertex; the whole lens moves:
    # the image distance (filter to sensor) grows by dz, chosen for the best field average
    s = system.copy()
    s.object_distance = distance
    s.max_field = distance*np.tan(system.max_field)   # object height: about the same image height
    best = None
    for dz in np.linspace(0.0, 0.3, 61):
        t = s.copy()
        t.surfaces[-1].thickness = system.surfaces[-1].thickness + dz
        with np.errstate(all="ignore"):
            w = np.mean([t.rms_opd(f*t.max_field, N=24) for f in (0, .5, .85)])
        if np.isfinite(w) and (best is None or w < best[0]):
            best = (w, dz, t)
    return best[2], best[1]


near, dz = close_focus(lens)
print("lens moved by {:.3f} mm for 100 mm (paraxial estimate f²/(s - f) = {:.3f} mm)".format(
    dz, pl.EFL**2/(100.0 + 0.0 - pl.EFL)))
Z_near = zernike_table(near, "Object at 100 mm (Fringe Zernikes Z4...Z16 [waves], rms without tilt)")

The same in a picture, the main terms over the field at the three
wavelengths:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.4), sharey=True)
for ax, Z, title in ((axes[0], Z_inf, "Object at infinity"), (axes[1], Z_near, "Object at 100 mm")):
    for wl, ls in zip(WLS, (":", "-", "--")):
        for j, col, lab in ((3, BLUE, "Z4 defocus"), (4, GREEN, "Z5 astigmatism"),
                            (7, ORANGE, "Z8 coma"), (8, RED, "Z9 spherical")):
            ax.plot(FIELDS, [Z[wl, f][0][j] for f in FIELDS], ls, color=col, lw=1.6,
                    label=lab if wl == WLS[1] else None)
        ax.plot(FIELDS, [Z[wl, f][1] for f in FIELDS], ls, color="k", lw=2.2,
                label="rms" if wl == WLS[1] else None)
    ax.set_xlabel("field")
    ax.set_title(title + "  (dotted 486, solid 588, dashed 656 nm)")
    ax.axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("[waves]")
axes[0].legend(fontsize=9)
fig.tight_layout()

## 8. Coda: Q2D freeform terms on a tilted mirror

Q-bfs is rotationally symmetric. Forbes extended it to freeforms (Q2D,
2012): the same best fit sphere and normal departure, with terms
$u^m Q_n^m(u^2)\cos m\varphi$ and $\sin m\varphi$ whose gradients are
orthonormal in the same weight, so the coefficients still add up to the
rms slope. A natural use next to a phone camera is the folded telephoto
("periscope") module, where a mirror or prism bends the beam by 90° and
there is room for freeform power in the fold.

A minimal example in the spirit of
notebook 155 (off-axis mirrors and nodal aberrations): a concave mirror, f = 30 mm at f/4, tilted by 12°,
over a ±1.5° field. The tilt brings astigmatism on axis. With
`qaspheres.Q2DSurface` and the coefficients of $m = 0, 2, 3$ (no $m = 1$
pure tilt, which would only re-aim the mirror) as variables:

In [ ]:
A_TILT = np.radians(12.0)
R_M, RN_M = -60.0, 4.0


def folded(cos=None, sin=None):
    S = [rt.Surface(np.inf, 20.0, semi_diameter=3.75),
         rt.CoordinateBreak(tilt=(A_TILT, 0, 0)),
         qa.Q2DSurface(R_M, 0.0, cos=cos, sin=sin, norm_radius=RN_M, mirror=True),
         rt.CoordinateBreak(tilt=(A_TILT, 0, 0), thickness=R_M/2)]
    return rt.System(S, 0, WLS[1], field_angle=np.radians(1.5))


F = np.radians(1.5)
fields2d = np.array([(0, 0), (F, 0), (-F, 0), (0, F), (0, -F), (F, F), (-F, -F), (F, -F), (-F, F)])
terms = [(0, 0, "cos"), (0, 1, "cos"), (1, 1, "cos"), (1, 1, "sin"), (1, 2, "sin"), (2, 0, "cos"),
         (2, 0, "sin"), (2, 1, "cos"), (2, 1, "sin"), (3, 0, "cos"), (3, 0, "sin"), (3, 1, "sin"),
         (4, 0, "cos")]
mirror0 = folded()
V = [qa.Q2DCoefficient(2, m, n, k) for m, n, k in terms] + [lo.Thickness(3)]
designs = {}
for name, fl in (("axis", fields2d[:1]), ("field", fields2d)):
    opt = lo.DampedLeastSquares(mirror0, V, [oa.field_wavefront(fl, rings=(0.5, 0.85, 1.0),
                                                                spokes=8, tolerance=0.02)])
    opt.run(max_iterations=40)
    designs[name] = opt.system
mirror1 = designs["axis"]
labels = ["axis", "+x", "-x", "+y", "-y", "+x+y", "-x-y", "+x-y", "-x+y"]
print("rms wavefront [waves]")
print("field     sphere    Q2D, optimized on axis    Q2D, optimized over the field")
for f, lab in zip(fields2d, labels):
    print("{:6s}    {:6.3f}    {:6.3f}                    {:6.3f}".format(
        lab, mirror0.rms_opd(tuple(f)), designs["axis"].rms_opd(tuple(f)), designs["field"].rms_opd(tuple(f))))
M1 = mirror1.surfaces[2]
print("Q2D coefficients of the on-axis design [µm]: " + ", ".join(
    "{}{}{}: {:+.2f}".format("a" if k == "cos" else "b", m, n, M1.get_term(k, m, n)*1e3)
    for m, n, k in terms if abs(M1.get_term(k, m, n)) > 1e-6)
    + "; rms slope {:.2f} mrad".format(M1.rms_slope()*1e3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
x = np.linspace(-RN_M, RN_M, 161)
X, Y = np.meshgrid(x, x)
inside = X**2 + Y**2 <= RN_M**2
D = np.where(inside, M1.departure(X, Y), np.nan)*1e3
im = axes[0].imshow(D, extent=[-RN_M, RN_M, -RN_M, RN_M], origin="lower", cmap="RdBu_r")
fig.colorbar(im, ax=axes[0], label="µm")
axes[0].set_title("Q2D departure of the mirror")
for ax, s, title in ((axes[1], mirror0, "tilted sphere"), (axes[2], mirror1, "tilted Q2D mirror")):
    W = s.opd((0.0, 0.0), N=64).W
    im = ax.imshow(W, extent=[-1, 1, -1, 1], origin="lower", cmap="RdBu_r")
    fig.colorbar(im, ax=ax, label="waves")
    ax.set_title("{}: wavefront on axis, rms {:.3f}".format(title, s.rms_opd((0.0, 0.0))))
fig.tight_layout()

The $m = 2$ term takes out the astigmatism of the tilt and the $m = 3$
terms the coma-like residue; what remains at the edge of the field is
the field dependence of a single surface
(notebook 155, off-axis mirrors and nodal aberrations), which a second freeform would address.

## Summary

- A five element f/1.8 phone lens with a 37° half field was designed
  from rough spherical radii with Q-bfs surfaces throughout, by widening
  the field in small steps; the chief ray angle limit had to grow with the
  field from early on.
- From the same start, the Q-bfs parameterization gives the damped least
  squares optimizer far better conditioned aspheric columns than
  monomials, whose neighbouring orders on the wavy rear surface are
  nearly collinear.
- The Q coefficients form a readable departure spectrum: the asphericity
  is concentrated on L4 and L5, and most of it in the first three orders.
- Manufacturing limits such as the rms slope or the high order content
  are simple quadratic functions of the Q coefficients and can be put on
  the optimizer directly, giving an explicit trade against image quality.
- Truncating Q-bfs terms degrades the lens gradually; truncating
  monomials does not.
- The final wavefront is given as Fringe Zernikes at 486, 588 and 656 nm
  for infinity and 100 mm.
- Q2D terms extend the description to freeforms; on a tilted mirror they
  remove the astigmatism of the tilt on axis.

## References

- G. W. Forbes, "Shape specification for axially symmetric optical
  surfaces", Opt. Express 15, 5218–5226 (2007).
- G. W. Forbes, "Manufacturability estimates for optical aspheres",
  Opt. Express 19, 9923–9941 (2011).
- G. W. Forbes, "Characterizing the shape of freeform optics", Opt.
  Express 20, 2483–2499 (2012).
- J. Sasián, "Introduction to Lens Design", Cambridge University Press
  (2019), on phone camera lenses.
- T. Steinich and V. Blahnik, "Optical design of camera optics for mobile
  phones", Adv. Opt. Technol. 1, 51–58 (2012).

In [ ]:
print("run time: {:.0f} s".format(time.time() - t_start))